# Plan #17: Re-Test Thesis with SOTA-Aligned Graph Rebuild

**Journey**: Rebuild MuSiQue graph with HippoRAG-aligned attributes, then prove GraphRAG solves questions that baseline can't.

**Purpose**: Produce decision-grade evidence for the adaptive-routing thesis. Primary metric: LLM-judge accuracy on baseline-failing questions.

**Mode**: Superseded — execution proceeded via CLI runs, not this notebook.
Results documented in `CURRENT_STATUS.md`. This notebook remains as a planning artifact showing the original phase contracts.

**Related docs**:
- Plan: `docs/plans/17_retest_thesis.md`
- ROADMAP: `docs/plans/ROADMAP.md`
- Literature review: `~/projects/investigations/digimon/2026-03-23-graphrag-sota-review.md`
- Failure diagnosis: `investigations/digimon/2026-03-23-musique-failure-diagnosis.md`

**Preliminary evidence** (before rebuild, consolidated tools only):
- Baseline: 21.1% LLM-judge (4/19)
- GraphRAG consolidated: 52.6% LLM-judge (10/19)
- Graph wins: 7 questions where baseline fails but GraphRAG succeeds

---
## Phase 1: Graph Rebuild with SOTA Config

**Input** → MuSiQue corpus (existing `results/MuSiQue/corpus/`), build config overrides  
**Output** → Rebuilt graph (`results/MuSiQue/er_graph/`), entity VDB, graph statistics  

**Acceptance criteria**:
- Graph has passage nodes (node_type="passage") alongside entity nodes
- Entity→passage edges exist (relation_name="extracted_from")
- Co-occurrence edges exist (relation_name="chunk_cooccurrence")
- Synonym edges added post-build
- Entity VDB contains only entity nodes (passage nodes filtered)
- PPR damping reads 0.5 from config

**Status**: `planned`  
**Execution mode**: `stub` (config defined, build not yet run)

In [ ]:
# Phase 1: Build configuration
# These are the config overrides for the SOTA-aligned rebuild.

BUILD_CONFIG = {
    # GraphConfig overrides
    "enable_passage_nodes": True,          # HippoRAG v2 bipartite graph
    "enable_chunk_cooccurrence": True,     # Entity co-occurrence edges
    "enable_entity_type": True,            # Typed entities
    "enable_entity_description": True,     # Entity descriptions
    "enable_edge_description": True,       # Edge descriptions
    "schema_mode": "open",                 # Schema-free extraction (HippoRAG-style)
    "skip_relationship_extraction": False,  # Keep relationships (test co-occurrence-only separately)
}

RETRIEVER_CONFIG = {
    "damping": 0.5,                        # HippoRAG PPR damping (was 0.1)
    "node_specificity": True,              # IDF-weighted PPR seeding
    "use_entity_similarity_for_ppr": False, # HippoRAG-style (not FastGraphRAG)
    "top_k": 10,
}

POST_BUILD_ENRICHMENT = [
    ("augment_synonym_edges", {"threshold": 0.85}),  # Synonym detection
    ("augment_centrality", {}),                        # PageRank centrality
]

DATASET = "MuSiQue"
ROUTING_MODEL = "openrouter/openai/gpt-5.4-mini"
EXTRACTION_MODEL = "gemini/gemini-2.5-flash"

print(f"Build config: {len(BUILD_CONFIG)} overrides")
print(f"Post-build: {len(POST_BUILD_ENRICHMENT)} enrichment steps")
print(f"Extraction: {EXTRACTION_MODEL}")
print(f"Routing: {ROUTING_MODEL}")

In [ ]:
# Phase 1 STUB: Graph rebuild (not yet executed)
# When promoted to live, this will:
# 1. Backup existing graph
# 2. Rebuild with BUILD_CONFIG
# 3. Run POST_BUILD_ENRICHMENT
# 4. Rebuild entity VDB
# 5. Verify passage nodes, co-occurrence edges, synonym edges

import os
from pathlib import Path

GRAPH_DIR = Path("results/MuSiQue/er_graph")
BACKUP_DIR = Path("results/MuSiQue/er_graph_pre_sota_backup")

# Provisional output: graph statistics from existing graph (pre-rebuild)
# This lets Phase 2 run with existing graph while rebuild is planned.
if GRAPH_DIR.exists():
    graphml = GRAPH_DIR / "nx_data.graphml"
    manifest = GRAPH_DIR / "graph_build_manifest.json"
    print(f"Existing graph: {graphml} ({graphml.stat().st_size / 1024:.0f} KB)")
    print(f"Manifest: {manifest.exists()}")
    
    # Count nodes/edges in existing graph
    import json
    if manifest.exists():
        with open(manifest) as f:
            m = json.load(f)
        print(f"Manifest: {json.dumps(m, indent=2)[:500]}")
else:
    print("No existing graph found — rebuild is mandatory")

# Provisional artifact for Phase 2
GRAPH_REBUILD_STATUS = "stub"  # will be "live" after rebuild
print(f"\nPhase 1 status: {GRAPH_REBUILD_STATUS}")

**Phase 1 gap**: Graph rebuild not yet executed. Existing graph lacks passage nodes, uses old PPR damping (0.1), no synonym edges. Phase 2 can run on existing graph for baseline comparison, but post-rebuild comparison is the primary deliverable.

**Promotion path**: `stub` → `live` when graph rebuild completes via CLI:
```bash
# 1. Backup
cp -r results/MuSiQue/er_graph results/MuSiQue/er_graph_pre_sota_backup

# 2. Rebuild (via benchmark runner or direct API)
# Set config overrides in Option/Config2.yaml or pass via tool call

# 3. Post-build enrichment
# augment_synonym_edges + augment_centrality via MCP tools

# 4. Rebuild VDB
# entity_vdb_build
```

---
## Phase 2: Baseline Comparison (19q diagnostic set)

**Input** → 19 MuSiQue question IDs (`eval/fixtures/musique_19q_diagnostic_ids.txt`), baseline results (already collected), graph (existing or rebuilt)  
**Output** → Cross-reference table: baseline vs GraphRAG per question, "graph wins" count  

**Acceptance criteria**:
- Baseline run completed on same 19 questions ✅ (21.1% LLM-judge)
- GraphRAG run completed on same 19 questions ✅ (52.6% LLM-judge, pre-rebuild)
- Cross-reference table produced with graph-wins / both-fail / regression counts
- After rebuild: re-run GraphRAG and compare

**Status**: `partial` (pre-rebuild comparison done, post-rebuild pending)  
**Execution mode**: `fixture` (using existing results as fixture for planning)

In [ ]:
# Phase 2: Cross-reference baseline vs GraphRAG (FIXTURE — using existing results)
import json
from collections import Counter

RESULT_FILES = {
    "baseline": "results/MuSiQue_gpt-5-4-mini_baseline_20260323T171234Z.json",
    "graphrag_old_prompt": "results/MuSiQue_gpt-5-4-mini_20260323T163655Z.json",
    "graphrag_consolidated": "results/MuSiQue_gpt-5-4-mini_consolidated_20260323T171612Z.json",
    # After rebuild, add:
    # "graphrag_rebuilt": "results/MuSiQue_gpt-5-4-mini_consolidated_YYYYMMDD.json",
}

results = {}
for name, path in RESULT_FILES.items():
    try:
        with open(path) as f:
            data = json.load(f)
        results[name] = {q["id"]: q for q in data["results"]}
        print(f"Loaded {name}: {len(results[name])} questions, LLM-judge={data.get('avg_llm_em', '?')}%")
    except FileNotFoundError:
        print(f"Missing: {name} ({path})")
        results[name] = {}

In [ ]:
# Phase 2: Cross-reference table
base = results.get("baseline", {})
best_graphrag = results.get("graphrag_consolidated", {}) or results.get("graphrag_old_prompt", {})

categories = Counter()
rows = []

for qid in sorted(base.keys(), key=lambda x: (x.split("__")[0], x)):
    b = base.get(qid, {}).get("llm_em", 0)
    g = best_graphrag.get(qid, {}).get("llm_em", 0)
    hops = qid.split("__")[0]
    
    if b == 0 and g == 1:
        cat = "GRAPH_WINS"
    elif b == 1 and g == 1:
        cat = "BOTH_PASS"
    elif b == 1 and g == 0:
        cat = "REGRESSION"
    else:
        cat = "BOTH_FAIL"
    
    categories[cat] += 1
    gold = base.get(qid, {}).get("gold", "?")[:40]
    pred = best_graphrag.get(qid, {}).get("predicted", "?")[:40]
    rows.append(f"{qid:<45} {hops:<6} B={b} G={g}  {cat:<12} gold={gold}")

print("=== CROSS-REFERENCE: Baseline vs GraphRAG (pre-rebuild) ===\n")
for row in rows:
    print(row)

print(f"\n=== SUMMARY ===")
for cat in ["GRAPH_WINS", "BOTH_PASS", "REGRESSION", "BOTH_FAIL"]:
    print(f"  {cat}: {categories[cat]}")
print(f"  Total: {sum(categories.values())}")
print(f"\n  Graph value signal: {categories['GRAPH_WINS']} wins, {categories['REGRESSION']} regressions")
print(f"  Iteration targets: {categories['BOTH_FAIL']} questions where both fail")

**Phase 2 gap**: Post-rebuild GraphRAG run not yet available. Current cross-reference uses pre-rebuild results. After Phase 1 completes, re-run GraphRAG on same 19q and update this comparison.

**Promotion path**: `fixture` → `live` when post-rebuild GraphRAG results are loaded into `RESULT_FILES["graphrag_rebuilt"]`.

---
## Phase 3: Failure Diagnosis on "Both Fail" Questions

**Input** → List of question IDs where both baseline and GraphRAG fail, graph instance, tool traces  
**Output** → Per-question diagnosis: where does the retrieval chain break? Is the answer in the graph?  

**Acceptance criteria**:
- Each "both fail" question has a diagnosis: answer in graph? entity found? PPR reaches it? chunk retrievable?
- Failures classified by family: extraction_miss / entity_linking_miss / graph_traversal_miss / retrieval_strategy / answer_synthesis
- Actionable recommendations for each failure family

**Status**: `planned`  
**Execution mode**: `planned` (diagnosis requires running tools interactively per question)

In [ ]:
# Phase 3: Diagnosis framework for "both fail" questions
# This defines the diagnostic protocol — execution happens per-question via tools.

DIAGNOSIS_PROTOCOL = """
For each "both fail" question:

1. DECOMPOSE: What are the sub-questions? What entities/facts are needed at each hop?
2. ENTITY CHECK: Is each answer-critical entity in the graph?
   - entity_search(method="string", query=entity_name) → found?
   - If not found: EXTRACTION_MISS
3. TRAVERSAL CHECK: Can PPR reach the target from the question entities?
   - entity_search(method="semantic", query=question) → seed entities
   - entity_traverse(method="ppr", entity_ids=seeds) → does target appear?
   - If not: GRAPH_TRAVERSAL_MISS (missing edge or edge weight too low)
4. CHUNK CHECK: Is the answer in a retrievable chunk?
   - chunk_retrieve(method="text", query_text=answer_fact) → found?
   - If not: CHUNK_MISS (chunking or indexing issue)
5. SYNTHESIS CHECK: Given the right evidence, does the LLM answer correctly?
   - reason(method="answer", query_text=question, context_chunks=[correct_chunks])
   - If wrong: ANSWER_SYNTHESIS failure
"""

FAILURE_FAMILIES = {
    "EXTRACTION_MISS": "Answer-critical entity not in graph. Fix: improve extraction prompt or add entity type.",
    "ENTITY_LINKING_MISS": "Entity in graph but not found by search. Fix: improve VDB embeddings or synonym edges.",
    "GRAPH_TRAVERSAL_MISS": "Entity found but PPR can't reach target. Fix: missing bridge edge, or edge weight too low.",
    "CHUNK_MISS": "Answer not in any retrievable chunk. Fix: chunking strategy or chunk indexing.",
    "RETRIEVAL_STRATEGY": "Evidence exists but agent uses wrong tool sequence. Fix: prompt or tool description.",
    "ANSWER_SYNTHESIS": "Evidence retrieved but LLM answers wrong. Fix: answer prompt or context formatting.",
}

# Provisional output: list of both-fail question IDs for diagnosis
both_fail_ids = [qid for qid in base if base[qid].get("llm_em", 0) == 0 
                 and best_graphrag.get(qid, {}).get("llm_em", 0) == 0]
print(f"Both-fail questions to diagnose: {len(both_fail_ids)}")
for qid in both_fail_ids:
    q = base[qid].get("question", "?")[:80]
    gold = base[qid].get("gold", "?")[:40]
    print(f"  {qid}: {q}")
    print(f"    Gold: {gold}")

**Phase 3 gap**: Diagnosis not yet executed. Requires interactive tool calls per question (use `/iterate-failures` skill). Post-rebuild diagnosis is more valuable than pre-rebuild since the graph will have passage nodes + synonym edges.

**Promotion path**: `planned` → `live` when each both-fail question has a classified failure family.

---
## Phase 4: Scale to 50q (conditional)

**Input** → Rebuilt graph, consolidated tools, 50 MuSiQue questions (seed=42)  
**Output** → Full benchmark comparison with statistical significance  

**Acceptance criteria**:
- GraphRAG LLM-judge > baseline LLM-judge by ≥5% on 50q
- Graph wins count documented
- Results compared to March 18 historical numbers (baseline 60%, fixed 54%, hybrid 44%)
- If criteria met: thesis evidence is decision-grade

**Status**: `planned`  
**Execution mode**: `planned` (only after Phase 2 post-rebuild shows positive signal)

**Gate**: Only proceed if Phase 2 post-rebuild shows ≥5 graph wins and ≤2 regressions on 19q.

---
## Phase 5: Evidence Summary

**Input** → All benchmark results, cross-reference tables, failure diagnoses  
**Output** → Investigation artifact documenting the thesis evidence  

**Acceptance criteria**:
- Clear statement: does the graph help? On which question types?
- Quantified: how many graph wins, at what cost, compared to baseline?
- Failure families classified with actionable next steps
- Decision: scale to 200q/1000q, or pivot?

**Status**: `planned`  
**Execution mode**: `planned`

In [ ]:
# Phase 5 STUB: Evidence summary template
EVIDENCE_TEMPLATE = """
# GraphRAG Thesis Evidence — MuSiQue {n_questions}q

## Result
- Baseline LLM-judge: {baseline_llm}%
- GraphRAG LLM-judge: {graphrag_llm}%
- Delta: {delta:+.1f}%
- Graph wins: {graph_wins} (baseline fails, GraphRAG succeeds)
- Regressions: {regressions} (baseline passes, GraphRAG fails)
- Both fail: {both_fail}

## Graph wins by hop count
{hop_breakdown}

## Failure families (both-fail questions)
{failure_families}

## Decision
{decision}

## Cost
- Graph build: ${build_cost:.2f}
- Per-question retrieval: ${per_q_cost:.4f}
- Total benchmark cost: ${total_cost:.2f}
"""

print("Evidence template ready. Will be filled after Phases 1-4 complete.")
print(f"\nNotebook status summary:")
print(f"  Phase 1 (Graph rebuild):     stub")
print(f"  Phase 2 (19q comparison):    fixture (pre-rebuild done)")
print(f"  Phase 3 (Failure diagnosis): planned")
print(f"  Phase 4 (50q scale):         planned (conditional)")
print(f"  Phase 5 (Evidence summary):  planned")